# Deep Learning Course
## a.y. 2025/2026, 27/03/2026

# SSL Hands-On: Representation Learning with PyTorch

## Empirical Risk Minimization (ERM) vs. Self-Supervised Learning (SSL)

In this lab, we will explore the power of Self-Supervised Learning (SSL) for learning visual representations without human-provided labels. We will use the **STL-10 dataset**, which is specifically designed for unsupervised learning (it has 100,000 unlabeled images and only 5,000 labeled images).

We will compare three paradigms:
1. **Plain ERM (Baseline):** Training a ResNet-18 from scratch on the small labeled dataset.
2. **Rotation Pretext Task:** A heuristic self-supervised method where the network learns to predict image rotations.
3. **SimCLR:** A state-of-the-art contrastive learning method that forces different augmented views of the same image to have similar representations.

### Introducing to Pytorch
Instead of Keras, today, we are using **PyTorch**. PyTorch is highly favored in research because it allows for immense flexibility.
* **Model Building:** Instead of `model.compile()`, we define custom `nn.Module` classes and explicitly write the forward pass.
* **The Training Loop:** Keras hides the training loop behind `model.fit()`. In PyTorch, you will write the training step yourself:
  1. `optimizer.zero_grad()` (clear old gradients)
  2. `loss.backward()` (compute new gradients)
  3. `optimizer.step()` (update weights).

### Imports and setting compute acceleration device

In [ ]:
import torch
from torchvision import datasets, transforms
from torch import nn
import random
import numpy as np
from torch.utils.data import DataLoader, TensorDataset, random_split
from matplotlib import pyplot as plt
from torchvision.models import resnet18
from torch.nn import functional as F
import os


print("CUDA available:", torch.cuda.is_available())
COMPUTE_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Performing computations on:", COMPUTE_DEVICE)

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


# For storing saved models
os.makedirs("models", exist_ok=True)
set_seed(1)

## Phase 1: Setup and Data Loading
Unlike Keras's `ImageDataGenerator`, PyTorch uses `Dataset` (to define how to load a single item) and `DataLoader` (to batch, shuffle, and parallelize data loading). We will use `transforms` to build augmentation pipelines.

### STL10 Dataset download

**This dataset has a bigger size than CIFAR10, we advise you to start downloading it as the very first step. It should take about 5 minutes to download it**

 + Here, we define the default transform that we will always apply to input images (`base_transform`), consisting in just square resizing and normalization in the range [0, 1].
 + First, we download the unlabeled training set: we will use a small subset of it, so training does not take too long.
 + Then, we download the labeled train set, that we will use for supervised training (either from scratch or fine-tuning).
 + From the train set, we hold-out a small portion to be used as validation set.
 + Finally, we download also the test set. It will be used to test the final classification models.

In [ ]:
base_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor()
])

batch_size = 256

unlabeled_trainset = datasets.STL10(root='./data', split="unlabeled", download=True, transform=base_transform)

# We randomly discard part of the unlabeled training set for time purposes
unlabeled_trainset, _ = random_split(unlabeled_trainset, [0.1, 0.9])

unlabeled_trainloader = DataLoader(unlabeled_trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2, drop_last=True)

train_set = datasets.STL10(root="./data", split="train", download=True, transform=base_transform)
train_set, val_set = random_split(train_set, [0.9, 0.1])


train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)
val_loader   = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2, drop_last=True)

test_set = datasets.STL10(root='./data', split="test",
                                       download=True, transform=base_transform)

test_loader = DataLoader(test_set, batch_size=batch_size,
                                         shuffle=False, num_workers=2, drop_last=True)

classes = ("airplane", "bird", "car", "cat", "deer", "dog", "horse", "monkey", "ship", "truck")

num_classes = 10

print("# Unlabeled Training Samples:", len(unlabeled_trainset))
print("# Labeled Training Samples", len(train_set))

### Visualize some images from both the unlabeled and the labeled training sets

In [ ]:
def show_samples(samples):
    plt.figure(figsize=(6, 3))
    for i, (x, y) in enumerate(samples):
        x = x.permute(1, 2, 0)
        plt.subplot(2, 5, i+1)
        plt.grid(False)
        plt.imshow(x)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(classes[y] if y != -1 else "Unlabeled")
    plt.tight_layout()
    plt.show()

In [ ]:
labeled_samples = [train_set[i] for i in torch.randint(0, len(train_set), size=(10, ))]
unlabeled_samples = [unlabeled_trainset[i] for i in torch.randint(0, len(unlabeled_trainset), size=(10, ))]

show_samples(labeled_samples)
show_samples(unlabeled_samples)

## SimCLR implementation with pytorch

We will implement a model for SimCLR using a python class inheriting from `nn.Module`. The fundamental methods that have to be implemented are:
+ `__init__`
    + Call to `super().__init__()` for initializing a base `nn.Module`.
    + Definition and initialization of whatever is needed for your particular model and objective.
+ `forward`
    + Depending on how you structured your model, you will define the flow of your input data through the model inside this method.  
    __NB__: calling `model.forward(...)` is equivalent to write `model(...)`.

In [ ]:
from torchvision.models import resnet18

class SimCLRModel(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # We set a randomly initialized ResNet-18 as backbone of our model (from scratch)
        self.encoder = resnet18(weights=None)
        self.embedding_dim = self.encoder.fc.in_features # this gets the number of features coming into the fc layer
        self.encoder.fc = nn.Identity()

        # We replace the original classification head with a non-linear projection, having a final embedding dimension of 256
        # you can play by changing this projection head and see if it impacts the performance
        self.fc = nn.Sequential(
            nn.Linear(self.embedding_dim, self.embedding_dim),
            nn.ReLU(),
            nn.Linear(self.embedding_dim, 256)
        )

        self.to(COMPUTE_DEVICE) # Enables GPU computations, if available

    def forward(self, x):
        x = self.encoder(x)
        return self.fc(x)

    def as_classification_model(self, num_classes):
        # store the pretrained_model for possible future uses
        torch.save(self.state_dict(), "./models/simclr_ssl_pretrained.pth")

        # swap the projection head with a classification layer with 'num_classes' output neurons
        self.fc = nn.Sequential(
            nn.ReLU(), #relu
            nn.Linear(512, num_classes) #linear layer with num_classes output neurons
        )

        self.to(COMPUTE_DEVICE)
        return self

    def save_model(self, model_id="saved_model"):
        torch.save(self.state_dict(), f"./models/{model_id}.pth")

    def load_model(self, model_id="saved_model"):
        self.load_state_dict(torch.load(f"./models/{model_id}.pth"))

    def restore_ssl_model(self):
        # store classification model, obtained after some supervised fine-tuning
        torch.save(self.state_dict(), "./models/simclr_ft_classification.pth")

        # reinitialize the original projection head
        self.fc = nn.Sequential(
            nn.Linear(self.embedding_dim, self.embedding_dim),
            nn.ReLU(),
            nn.Linear(self.embedding_dim, 256)
        )

        # reload the previously stored weights
        try:
            self.load_state_dict(torch.load("./models/simclr_ssl_pretrained.pth"))
        except FileNotFoundError:
            print("Unable to find stored model in './models/simclr_ssl_pretrained.pth', keeping model as it is.")
            return self

        return self

### Define the transform function that we will use during contrastive learning

In order, define the following transform inside a `transforms.Compose` object:
    
1. RandomHorizontalFlip.
2. RandomResizedCrop to a square size of 96x96 pixel.
3. Random application of ColorJitter. This transform will be applied with a probability of 0.8. (see transforms.RandomApply())
4. GaussianBlur with a kernel size of 9.
5. RandomGrayscale with a probability of 0.2.
6. Normalization with `mean=[0.5, 0.5, 0.5]` and `std=[0.5, 0.5, 0.5]`.

In [ ]:
# This function returns a Torchvision Transform that we will use to apply augmentations to input images
def build_transforms_for_contrastive():
    return transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomResizedCrop(size=96),
        transforms.Lambda(lambda x : torch.clamp(x, min=0.0, max=1.0)),  #this line is needed because color jitter does not support negative input values
        transforms.RandomApply([transforms.ColorJitter(brightness=0.8, contrast=0.8, saturation=0.8, hue=0.2)], p=0.8),
        transforms.GaussianBlur(kernel_size=9),
        transforms.RandomGrayscale(p=0.2),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

### Complete the definition of the InfoNCE loss function.
We will implement it as a `nn.Module`.
Again we will need to define the `__init__` and `forward` methods.

The forward will contain the actual loss computation, while the constructor will serve to set the temperature parameter $\tau$.

Consider that the InfoNCE loss can be written as:

$$-\log\frac{\exp(sim(z_i, z_j) / \tau)}{\sum_{k=1}^{2N}{1_{[k \neq i]}{\exp(sim(z_i, z_k)/\tau)}}} = -sim(z_i, z_j) / \tau + \log \left( \sum_{k=1}^{2N}{1_{[k \neq i]}{\exp(sim(z_i, z_k)/\tau)}} \right)
$$

In [ ]:
class InfoNCELoss(nn.Module):
    def __init__(self, tau, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.tau = tau

    def forward(self, z):
        # Computes the cosine similarity between each pair of embeddings
        z_normalized = F.normalize(z, dim=1)
        cos_sim = torch.matmul(z_normalized, z_normalized.T)

        # Create a mask that selects the similarities between each element and itself
        self_mask = torch.eye(cos_sim.size(0), dtype=torch.bool, device=cos_sim.device)

        # Fills cos_sim with -torch.inf in the entries selected with self_mask
        cos_sim = cos_sim.masked_fill(self_mask, -9e15)

        # Creates a mask that for each sample selects its positive pair
        # Trick: Consider that by concatenating z1 and z2 into z
        #       each z1[i] has its positive sample z2[i] in z[i + batch_size].
        #       Keep in mind that for a z2[i], z1[i] is in z[i - batch_size].
        #       (Look for the 'roll' method in torch docs)
        pos_mask = self_mask.roll(cos_sim.size(0) // 2, dims=0)

        # scale cosine similarity by temperature parameter tau
        cos_sim = torch.div(cos_sim, self.tau)

        # Leverage the simplified loss expression (look for the logsumexp operation)
        neglog_likelihood = -cos_sim[pos_mask] + torch.logsumexp(cos_sim, dim=-1)

        # returns the average
        return neglog_likelihood.mean()

### Function to initialize a fresh optimizer.

In [ ]:
def init_optimizer(model, learning_rate=0.0005):
    return torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

## SimCLR training loop

In [ ]:
def simclr_train(model, optimizer, num_epochs, criterion):
    tr_losses = []

    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_epochs, eta_min=1e-12)
    contrastive_transform = build_transforms_for_contrastive()

    for epoch in range(num_epochs):
        model.train()

        with torch.enable_grad():
            running_tr_loss = 0.0
            for i, (x, _) in enumerate(unlabeled_trainloader):      # We discard y using '_', as we do not need it
                x = x.to(COMPUTE_DEVICE)                            # Offloads data to GPU's VRAM (or keeps everything on CPU+RAM if a GPU is not available)
                x1 = contrastive_transform(x)
                x2 = contrastive_transform(x)


                # Concatenate x1 and x2 into a new tensor x of size
                x = torch.cat([x1, x2], dim=0)

                # Zero gradients in the optimizer
                optimizer.zero_grad()


                z = model(x)

                loss = criterion(z)

                loss.backward()                              # Compute gradients with backprop
                optimizer.step()                              # Update model weights

                running_tr_loss += loss.item()          # Update running loss

        avg_tr_loss = running_tr_loss / len(unlabeled_trainloader)
        tr_losses.append(avg_tr_loss)

        lr_scheduler.step()
        print(f"[Epoch {epoch + 1} / {num_epochs}]: Train Loss: {avg_tr_loss:.3f}")

    return tr_losses

### Training loop for supervised learning on the labeled training set.

In [ ]:
def supervised_train(model, optimizer, num_epochs, criterion):
    tr_losses = []
    vl_losses = []

    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode="min", factor=0.9, patience=2)

    for epoch in range(num_epochs):  # loop over the dataset multiple times
        running_tr_loss = 0.0
        tr_correct_preds = 0
        num_samples   = 0
        model.train() # sets model in train mode
        with torch.enable_grad():
            for i, data in enumerate(train_loader): # get a batch of (inputs, labels); train_loader is an iterable of batches
                inputs, labels = data

                # load inputs and labels to GPU
                inputs = inputs.to(COMPUTE_DEVICE)
                labels = labels.to(COMPUTE_DEVICE)

                num_samples += labels.size(0)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward pass
                outputs = model(inputs)

                # compute loss as criterion(outputs, labels)
                loss = criterion(outputs, labels)

                # collect predictions using torch.max
                _, preds = torch.max(outputs, dim=1)

                # accumulate number of correct predictions
                tr_correct_preds += torch.count_nonzero(preds == labels).item()

                # backward pass
                loss.backward()

                # perform the otpimization step
                optimizer.step()

                # print statistics
                running_tr_loss += loss.item()

        avg_tr_loss = running_tr_loss / len(train_loader)
        tr_accuracy = tr_correct_preds / num_samples
        tr_losses.append(avg_tr_loss)


        running_vl_loss  = 0.0
        vl_correct_preds = 0
        num_samples   = 0

        model.eval()
        with torch.no_grad():
            for i, data in enumerate(val_loader): # get a batch of (inputs, labels); val_loader is an iterable of batches
                inputs, labels = data
                inputs = inputs.to(COMPUTE_DEVICE)
                labels = labels.to(COMPUTE_DEVICE)

                num_samples += labels.size(0)
                # zero the parameter gradients
                optimizer.zero_grad()

                # forward + backward + optimize
                outputs = model(inputs)
                _, preds = torch.max(outputs, dim=1)
                vl_correct_preds += torch.count_nonzero(preds == labels).item()
                loss = criterion(outputs, labels)
                # print statistics
                running_vl_loss += loss.item()

        avg_vl_loss = running_vl_loss / len(val_loader)
        vl_accuracy  = vl_correct_preds / num_samples
        vl_losses.append(avg_vl_loss)

        lr_scheduler.step(avg_vl_loss)
        print(f"[Epoch {epoch + 1} / {num_epochs}]: ")
        print(f"\t -- Train Loss: {avg_tr_loss:.3f}, Train Accuracy: {tr_accuracy:.3f}")
        print(f"\t -- Validation Loss: {avg_vl_loss:.3f}, Validation Accuracy: {vl_accuracy:.3f}")

    return tr_losses, vl_losses

### Function to evaluate your classification model on the test set

In [ ]:
def supervised_test(model, test_loader, criterion):
    model.eval()
    correct_preds = 0
    num_samples   = 0
    running_loss  = 0.0
    with torch.no_grad():
        for i, data in enumerate(test_loader): # get a batch of (inputs, labels); test_loader is an iterable of batches
            inputs, labels = data
            inputs = inputs.to(COMPUTE_DEVICE) # to device
            labels = labels.to(COMPUTE_DEVICE) # to device

            num_samples += labels.size(0)

            outputs = model(inputs)
            _, preds = torch.max(outputs, dim=1) #argmax of the output with torch.max

            correct_preds += torch.count_nonzero(preds == labels).item() #number of corrected predictions

            loss = criterion(outputs, labels)
            # print statistics
            running_loss += loss.item()

    mean_loss = running_loss / len(test_loader)
    accuracy  = correct_preds / num_samples
    print(f'Loss: {mean_loss:.3f}, accuracy: {accuracy:.3f}')

In [ ]:
scratch_model = SimCLRModel()
scratch_model.as_classification_model(num_classes=10)
optimizer = init_optimizer(scratch_model, learning_rate=0.0001)
tr_losses, vl_losses = supervised_train(scratch_model, optimizer, 10, nn.CrossEntropyLoss())
scratch_model.save_model(model_id="scratch")
scratch_model.load_state_dict(torch.load("./models/scratch.pth"))
supervised_test(scratch_model, test_loader, nn.CrossEntropyLoss())

In [ ]:
simclr_model = SimCLRModel()
optimizer = init_optimizer(simclr_model, learning_rate=0.0005)
criterion = InfoNCELoss(tau=0.7)
tr_losses = simclr_train(simclr_model, optimizer, 10, criterion)
simclr_model.save_model(model_id="ssl_pretrain")

In [ ]:
simclr_model.as_classification_model(num_classes=10)
optimizer = init_optimizer(simclr_model, learning_rate=0.0001)
tr_losses, vl_losses = supervised_train(simclr_model, optimizer, 10, nn.CrossEntropyLoss())
supervised_test(simclr_model, test_loader, nn.CrossEntropyLoss())
simclr_model.save_model(model_id="ssl_finetuned")

### Rotation Pretext Task

Here we report again the most important code blocks from the last hands on activity. You can reuse it to perform the final comparison.

In this lab, we do not need to build a subset of the training set, as STL-10 already provides the unlabeled split for pretext tasks and we already have our DataLoader for accessing batches of images.  
Therefore, use `unlabeled_trainloader` for the SSL pretraining.

In [ ]:
def rotate(images):
    y = torch.randint(0, 4, size=(len(images), ))
    for i, img in enumerate(images):
        images[i] = transforms.functional.rotate(img, int(y[i]) * 90)

    return images, y

# empty tensors to store our dataset
x_rot = torch.zeros(len(unlabeled_trainloader) * batch_size, 3, 96, 96)
y_rot = torch.zeros(len(unlabeled_trainloader) * batch_size)

#let us fill x and y with the rotate function
for i, data in enumerate(unlabeled_trainloader):
    inputs, labels = data
    x_rot[i*batch_size : i*batch_size + batch_size], y_rot[i * batch_size : i*batch_size + batch_size] = rotate(inputs)

train_set_rotation = TensorDataset(x_rot, y_rot)
train_loader_rotation = DataLoader(train_set_rotation, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

def build_model(freeze_backbone = False, num_classes = 10, weights_path=None):

    model = resnet18(weights=None)

    if freeze_backbone == True:
        for param in model.parameters():
            param.requires_grad = False

    else:
       for param in model.parameters():
            param.requires_grad = True

    if weights_path==None:

        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)

    elif 'scratch' in weights_path or 'imagenet' in weights_path:


        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
        model.load_state_dict(torch.load(weights_path))


    elif 'rot' in weights_path:


        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, 4)
        model.load_state_dict(torch.load(weights_path))
        model.fc = nn.Linear(num_features, num_classes)

    else :

        model.load_state_dict(torch.load(weights_path),strict=False)
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)

    model.fc.requires_grad_ = True

    model.to(COMPUTE_DEVICE) # loads model to the first available GPU

    return model

#Final test

As a final test, you can take the code for the rotation pre-text task from the previous lab, following the comments in the cell below.
Make final comments on the difference between a basic SSL pretext task as rotation, and simCLR, while also pointing out the differences between pre-trainings in terms of test accuracy.

In [ ]:
# train rotation on the unlabaled training set
rotation_model = build_model(num_classes=4)
optimizer = init_optimizer(rotation_model, learning_rate=0.0005)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    rotation_model.train()
    running_loss = 0.0
    for i, data in enumerate(train_loader_rotation):
        inputs, labels = data
        inputs = inputs.to(COMPUTE_DEVICE)
        labels = labels.to(COMPUTE_DEVICE)
        
        optimizer.zero_grad()
        outputs = rotation_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"[Epoch {epoch + 1} / 10]: Loss: {running_loss / len(train_loader_rotation):.3f}")

torch.save(rotation_model.state_dict(), "./models/rot_pretrain.pth")

# fine-tune the model on the labaled training set
rotation_ft_model = build_model(num_classes=10, weights_path="./models/rot_pretrain.pth")
optimizer = init_optimizer(rotation_ft_model, learning_rate=0.0001)
tr_losses, vl_losses = supervised_train(rotation_ft_model, optimizer, 10, nn.CrossEntropyLoss())
supervised_test(rotation_ft_model, test_loader, nn.CrossEntropyLoss())

# compare scratch, rotation and simCLR.
print("\n=== Final Comparison ===")
print("1. Scratch Model (Supervised only):")
scratch_model = build_model(num_classes=10, weights_path="./models/scratch.pth")
supervised_test(scratch_model, test_loader, nn.CrossEntropyLoss())

print("\n2. Rotation Pretext Task + Fine-tuning:")
rotation_ft_model = build_model(num_classes=10, weights_path="./models/rot_pretrain.pth")
supervised_test(rotation_ft_model, test_loader, nn.CrossEntropyLoss())

print("\n3. SimCLR + Fine-tuning:")
simclr_model = SimCLRModel()
simclr_model.as_classification_model(num_classes=10)
simclr_model.load_state_dict(torch.load("./models/ssl_finetuned.pth"))
supervised_test(simclr_model, test_loader, nn.CrossEntropyLoss())